# 🛡️ FraudShield AI

## Business Problem

FraudShield AI is a decision-support system for identifying potentially fraudulent financial transactions.

The model estimates fraud risk so that suspicious transactions can be prioritized for review.

Two types of errors are especially important:

- **False Negative:** a fraudulent transaction is classified as legitimate, potentially causing financial loss.
- **False Positive:** a legitimate transaction is flagged as fraud, increasing investigation workload and potentially affecting customer experience.

The objective is therefore not to maximize accuracy, but to balance fraud recall, precision, false-positive workload, and missed-fraud risk.

In [176]:
import pandas as pd
import numpy as np

In [177]:
file_path = "/kaggle/input/datasets/ealaxi/paysim1/PS_20174392719_1491204439457_log.csv"

df = pd.read_csv(file_path)

# **PHASE 1 - Data Audit**

In [178]:
df.shape

(6362620, 11)

In [179]:
df.columns

Index(['step', 'type', 'amount', 'nameOrig', 'oldbalanceOrg', 'newbalanceOrig',
       'nameDest', 'oldbalanceDest', 'newbalanceDest', 'isFraud',
       'isFlaggedFraud'],
      dtype='object')

In [180]:
df.head()

,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
0,1,PAYMENT,9839.64,C1231006815,170136.0,160296.36,M1979787155,0.0,0.0,0,0
1,1,PAYMENT,1864.28,C1666544295,21249.0,19384.72,M2044282225,0.0,0.0,0,0
2,1,TRANSFER,181.00,C1305486145,181.0,0.00,C553264065,0.0,0.0,1,0
3,1,CASH_OUT,181.00,C840083671,181.0,0.00,C38997010,21182.0,0.0,1,0
4,1,PAYMENT,11668.14,C2048537720,41554.0,29885.86,M1230701703,0.0,0.0,0,0


In [181]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6362620 entries, 0 to 6362619
Data columns (total 11 columns):
 #   Column          Dtype  
---  ------          -----  
 0   step            int64  
 1   type            object 
 2   amount          float64
 3   nameOrig        object 
 4   oldbalanceOrg   float64
 5   newbalanceOrig  float64
 6   nameDest        object 
 7   oldbalanceDest  float64
 8   newbalanceDest  float64
 9   isFraud         int64  
 10  isFlaggedFraud  int64  
dtypes: float64(5), int64(3), object(3)
memory usage: 534.0+ MB


### Check null values

In [182]:
df.isnull().sum()

step              0
type              0
amount            0
nameOrig          0
oldbalanceOrg     0
newbalanceOrig    0
nameDest          0
oldbalanceDest    0
newbalanceDest    0
isFraud           0
isFlaggedFraud    0
dtype: int64

### Check Duplicates

In [183]:
df.duplicated().sum()

np.int64(0)

### Transaction types

In [184]:
df["type"].value_counts()

type
CASH_OUT    2237500
PAYMENT     2151495
CASH_IN     1399284
TRANSFER     532909
DEBIT         41432
Name: count, dtype: int64

### Fraud distribution

In [185]:
fraud_counts = df["isFraud"].value_counts()
fraud_percentages = df["isFraud"].value_counts(normalize=True) * 100

print("Counts:")
print(fraud_counts)

print("\nPercentages:")
print(fraud_percentages)

Counts:
isFraud
0    6354407
1       8213
Name: count, dtype: int64

Percentages:
isFraud
0    99.870918
1     0.129082
Name: proportion, dtype: float64


In [186]:
pd.crosstab(df["type"], df["isFraud"])

isFraud,0,1
type,,
CASH_IN,1399284,0
CASH_OUT,2233384,4116
DEBIT,41432,0
PAYMENT,2151495,0
TRANSFER,528812,4097


In [187]:
fraud_by_type = (
    df.groupby("type")["isFraud"]
    .agg(["count", "sum", "mean"])
    .sort_values("mean", ascending=False)
)

fraud_by_type

,count,sum,mean
type,,,
TRANSFER,532909,4097,0.007688
CASH_OUT,2237500,4116,0.001840
CASH_IN,1399284,0,0.000000
DEBIT,41432,0,0.000000
PAYMENT,2151495,0,0.000000


In [188]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
step,6362620.0,2.433972e+02,1.423320e+02,1.0,156.00,239.000,3.350000e+02,7.430000e+02
amount,6362620.0,1.798619e+05,6.038582e+05,0.0,13389.57,74871.940,2.087215e+05,9.244552e+07
oldbalanceOrg,6362620.0,8.338831e+05,2.888243e+06,0.0,0.00,14208.000,1.073152e+05,5.958504e+07
newbalanceOrig,6362620.0,8.551137e+05,2.924049e+06,0.0,0.00,0.000,1.442584e+05,4.958504e+07
oldbalanceDest,6362620.0,1.100702e+06,3.399180e+06,0.0,0.00,132705.665,9.430367e+05,3.560159e+08
newbalanceDest,6362620.0,1.224996e+06,3.674129e+06,0.0,0.00,214661.440,1.111909e+06,3.561793e+08
isFraud,6362620.0,1.290820e-03,3.590480e-02,0.0,0.00,0.000,0.000000e+00,1.000000e+00
isFlaggedFraud,6362620.0,2.514687e-06,1.585775e-03,0.0,0.00,0.000,0.000000e+00,1.000000e+00


In [189]:
invalid_value_audit = pd.DataFrame({
    "Check": [
        "Negative transaction amounts",
        "Missing transaction types",
        "Missing target labels",
        "Unexpected target values"
    ],
    "Count": [
        (df["amount"] < 0).sum(),
        df["type"].isna().sum(),
        df["isFraud"].isna().sum(),
        (~df["isFraud"].isin([0, 1])).sum()
    ]
})

invalid_value_audit

,Check,Count
0,Negative transaction amounts,0
1,Missing transaction types,0
2,Missing target labels,0
3,Unexpected target values,0


# **PHASE 2 - Leakage decisions**
## Feature Selection and Leakage Considerations

The target variable is `isFraud`.

The following fields are excluded from the initial model:

- `nameOrig` and `nameDest`: account identifiers rather than meaningful generalizable behavioral features.
- `isFlaggedFraud`: an existing fraud-rule output that may leak information from another detection mechanism.
- `newbalanceOrig` and `newbalanceDest`: post-transaction balances that may introduce information unavailable or inappropriate at prediction time.

The initial model therefore uses transaction information that can reasonably represent the transaction at scoring time.

In [190]:
leakage_review = pd.DataFrame({
    "Feature": [
        "nameOrig",
        "nameDest",
        "isFlaggedFraud",
        "newbalanceOrig",
        "newbalanceDest"
    ],
    "Reason for Exclusion": [
        "Identifier - unlikely to generalize",
        "Identifier - unlikely to generalize",
        "Existing rule-based fraud signal - potential leakage",
        "Post-transaction balance - potential leakage",
        "Post-transaction balance - potential leakage"
    ]
})

leakage_review

,Feature,Reason for Exclusion
0,nameOrig,Identifier - unlikely to generalize
1,nameDest,Identifier - unlikely to generalize
2,isFlaggedFraud,Existing rule-based fraud signal - potential l...
3,newbalanceOrig,Post-transaction balance - potential leakage
4,newbalanceDest,Post-transaction balance - potential leakage


In [191]:
selected_features = [
    "step",
    "type",
    "amount",
    "oldbalanceOrg",
    "oldbalanceDest"
]

excluded_features = [
    col for col in df.columns
    if col not in selected_features + ["isFraud"]
]

print("Selected features:")
print(selected_features)

print("\nExcluded features:")
print(excluded_features)

Selected features:
['step', 'type', 'amount', 'oldbalanceOrg', 'oldbalanceDest']

Excluded features:
['nameOrig', 'newbalanceOrig', 'nameDest', 'newbalanceDest', 'isFlaggedFraud']


In [192]:
feature_dictionary = pd.DataFrame({
    "Feature": [
        "step",
        "type",
        "amount",
        "nameOrig",
        "oldbalanceOrg",
        "newbalanceOrig",
        "nameDest",
        "oldbalanceDest",
        "newbalanceDest",
        "isFraud",
        "isFlaggedFraud"
    ],

    "Type": [
        "Numeric",
        "Categorical",
        "Numeric",
        "Identifier",
        "Numeric",
        "Numeric",
        "Identifier",
        "Numeric",
        "Numeric",
        "Target",
        "Rule-based flag"
    ],

    "Description": [
        "Simulation time step",
        "Transaction category",
        "Transaction amount",
        "Origin account identifier",
        "Origin balance before transaction",
        "Origin balance after transaction",
        "Destination account identifier",
        "Destination balance before transaction",
        "Destination balance after transaction",
        "Ground-truth fraud label",
        "Existing rule-based fraud flag"
    ],

    "Model Use": [
        "Included",
        "Included",
        "Included",
        "Excluded - identifier",
        "Included",
        "Excluded - potential leakage",
        "Excluded - identifier",
        "Included",
        "Excluded - potential leakage",
        "Target",
        "Excluded - potential leakage"
    ]
})

feature_dictionary

,Feature,Type,Description,Model Use
0,step,Numeric,Simulation time step,Included
1,type,Categorical,Transaction category,Included
2,amount,Numeric,Transaction amount,Included
3,nameOrig,Identifier,Origin account identifier,Excluded - identifier
4,oldbalanceOrg,Numeric,Origin balance before transaction,Included
5,newbalanceOrig,Numeric,Origin balance after transaction,Excluded - potential leakage
6,nameDest,Identifier,Destination account identifier,Excluded - identifier
7,oldbalanceDest,Numeric,Destination balance before transaction,Included
8,newbalanceDest,Numeric,Destination balance after transaction,Excluded - potential leakage
9,isFraud,Target,Ground-truth fraud label,Target


# **PHASE 3 - Create modeling dataset**

### Working Sample Rationale

The full PaySim dataset contains more than 6.3 million transactions.

To keep training and experimentation computationally manageable within the available Kaggle environment and project deadline, a stratified working sample of 800,000 transactions was selected.

Stratification preserved the original fraud proportion so that the sampled data retained the severe class imbalance present in the full dataset.

In [193]:
features = [
    "step",
    "type",
    "amount",
    "oldbalanceOrg",
    "oldbalanceDest"
]

X = df[features]
y = df["isFraud"]

print(X.shape)
print(y.shape)

(6362620, 5)
(6362620,)


In [194]:
from sklearn.model_selection import train_test_split

X_work, _, y_work, _ = train_test_split(
    X,
    y,
    train_size=800_000,
    stratify=y,
    random_state=42
)

print("Working sample shape:", X_work.shape)
print("\nFraud distribution:")
print(y_work.value_counts())
print("\nFraud percentage:")
print(y_work.value_counts(normalize=True) * 100)

Working sample shape: (800000, 5)

Fraud distribution:
isFraud
0    798967
1      1033
Name: count, dtype: int64

Fraud percentage:
isFraud
0    99.870875
1     0.129125
Name: proportion, dtype: float64


# **PHASE 4 - Train / validation / test**

In [195]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X_work,
    y_work,
    test_size=0.30,
    stratify=y_work,
    random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    stratify=y_temp,
    random_state=42
)

print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)

print("\nFraud rates:")
print("Train:", y_train.mean())
print("Validation:", y_val.mean())
print("Test:", y_test.mean())

Train: (560000, 5)
Validation: (120000, 5)
Test: (120000, 5)

Fraud rates:
Train: 0.0012910714285714285
Validation: 0.0012916666666666667
Test: 0.0012916666666666667


# **PHASE 5 - Preprocessing**

In [196]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

numeric_features = [
    "step",
    "amount",
    "oldbalanceOrg",
    "oldbalanceDest"
]

categorical_features = ["type"]

preprocessor = ColumnTransformer(
    transformers=[
        ("num",StandardScaler(),numeric_features),
        ("cat",OneHotEncoder(handle_unknown="ignore"),categorical_features)
    ])

preprocessor

ColumnTransformer(transformers=[('num', StandardScaler(),
                                 ['step', 'amount', 'oldbalanceOrg',
                                  'oldbalanceDest']),
                                ('cat', OneHotEncoder(handle_unknown='ignore'),
                                 ['type'])])

### Data Preprocessing

The preprocessing workflow is fitted only on the training data to avoid data leakage.

- `fit_transform()` is applied to the training set so the preprocessing steps learn only from the training data.
- `transform()` is applied to the validation and test sets using the same transformations learned from the training set.
- After preprocessing, the feature space increases from the original selected features to **9 processed features** due to categorical encoding.

This ensures a leakage-safe and consistent preprocessing workflow across the train, validation, and test datasets.

In [197]:
X_train_processed = preprocessor.fit_transform(X_train)
X_val_processed = preprocessor.transform(X_val)
X_test_processed = preprocessor.transform(X_test)

print("Train processed:", X_train_processed.shape)
print("Validation processed:", X_val_processed.shape)
print("Test processed:", X_test_processed.shape)

Train processed: (560000, 9)
Validation processed: (120000, 9)
Test processed: (120000, 9)


# **PHASE 6 - Baseline model**

In this phase, a baseline machine learning model is trained to establish an initial performance benchmark for fraud detection.

The purpose of the baseline model is to:
- provide a reference point for future model comparisons,
- evaluate how well a simple model can detect fraudulent transactions,
- identify the effect of class imbalance on model performance,
- and create a benchmark before applying more advanced models or optimization techniques.

Because fraud detection is highly imbalanced, evaluation will not rely only on accuracy. Metrics such as **Precision, Recall, F1-score, ROC-AUC, and PR-AUC** will be considered to better assess fraud detection performance.

In [198]:
from sklearn.linear_model import LogisticRegression

logistic_model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced",
    random_state=42
)

logistic_model.fit(X_train_processed, y_train)

LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)

In [199]:
y_val_pred_lr = logistic_model.predict(X_val_processed)
y_val_prob_lr = logistic_model.predict_proba(X_val_processed)[:, 1]

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    average_precision_score
)

y_val_pred_lr = logistic_model.predict(X_val_processed)
y_val_prob_lr = logistic_model.predict_proba(X_val_processed)[:, 1]

print("LOGISTIC REGRESSION - VALIDATION RESULTS")
print()

print(classification_report(y_val, y_val_pred_lr))

print("\nROC-AUC:",
      roc_auc_score(y_val, y_val_prob_lr))

print("PR-AUC:",
      average_precision_score(y_val, y_val_prob_lr))

LOGISTIC REGRESSION - VALIDATION RESULTS

              precision    recall  f1-score   support

           0       1.00      0.92      0.96    119845
           1       0.02      0.93      0.03       155

    accuracy                           0.92    120000
   macro avg       0.51      0.93      0.50    120000
weighted avg       1.00      0.92      0.96    120000


ROC-AUC: 0.978334838413596
PR-AUC: 0.24745826720202338


In [200]:
cm_lr = confusion_matrix(y_val, y_val_pred_lr)

cm_lr_df = pd.DataFrame(
    cm_lr,
    index=["Actual Legitimate", "Actual Fraud"],
    columns=["Predicted Legitimate", "Predicted Fraud"]
)

cm_lr_df

,Predicted Legitimate,Predicted Fraud
Actual Legitimate,110632,9213
Actual Fraud,11,144


# **PHASE 7 - Model 2: Decision Tree**

In [201]:
from sklearn.tree import DecisionTreeClassifier

decision_tree_model = DecisionTreeClassifier(
    max_depth=12,
    min_samples_leaf=10,
    class_weight="balanced",
    random_state=42
)

decision_tree_model.fit(X_train_processed, y_train)

DecisionTreeClassifier(class_weight='balanced', max_depth=12,
                       min_samples_leaf=10, random_state=42)

In [202]:
y_val_pred_dt = decision_tree_model.predict(X_val_processed)
y_val_prob_dt = decision_tree_model.predict_proba(X_val_processed)[:, 1]

print("DECISION TREE - VALIDATION RESULTS")
print()

print(classification_report(y_val, y_val_pred_dt))

print("\nROC-AUC:",
      roc_auc_score(y_val, y_val_prob_dt))

print("PR-AUC:",
      average_precision_score(y_val, y_val_prob_dt))

DECISION TREE - VALIDATION RESULTS

              precision    recall  f1-score   support

           0       1.00      0.97      0.98    119845
           1       0.03      0.94      0.07       155

    accuracy                           0.97    120000
   macro avg       0.52      0.95      0.52    120000
weighted avg       1.00      0.97      0.98    120000


ROC-AUC: 0.9678986163579569
PR-AUC: 0.6814711835093367


In [203]:
cm_dt = confusion_matrix(y_val, y_val_pred_dt)

cm_dt_df = pd.DataFrame(
    cm_dt,
    index=["Actual Legitimate", "Actual Fraud"],
    columns=["Predicted Legitimate", "Predicted Fraud"]
)

cm_dt_df

,Predicted Legitimate,Predicted Fraud
Actual Legitimate,115736,4109
Actual Fraud,9,146


# **PHASE 8 - Model 3: Random Forest**

In [204]:
from sklearn.ensemble import RandomForestClassifier

random_forest_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=15,
    min_samples_leaf=5,
    class_weight="balanced",
    n_jobs=-1,
    random_state=42
)

random_forest_model.fit(X_train_processed, y_train)

RandomForestClassifier(class_weight='balanced', max_depth=15,
                       min_samples_leaf=5, n_jobs=-1, random_state=42)

In [205]:
y_val_pred_rf = random_forest_model.predict(X_val_processed)
y_val_prob_rf = random_forest_model.predict_proba(X_val_processed)[:, 1]

print("RANDOM FOREST - VALIDATION RESULTS")
print()

print(classification_report(y_val, y_val_pred_rf))

print("\nROC-AUC:",
      roc_auc_score(y_val, y_val_prob_rf))

print("PR-AUC:",
      average_precision_score(y_val, y_val_prob_rf))

RANDOM FOREST - VALIDATION RESULTS

              precision    recall  f1-score   support

           0       1.00      0.99      1.00    119845
           1       0.16      0.83      0.27       155

    accuracy                           0.99    120000
   macro avg       0.58      0.91      0.63    120000
weighted avg       1.00      0.99      1.00    120000


ROC-AUC: 0.9921215440912252
PR-AUC: 0.7480476747439795


In [206]:
cm_rf = confusion_matrix(y_val, y_val_pred_rf)

cm_rf_df = pd.DataFrame(
    cm_rf,
    index=["Actual Legitimate", "Actual Fraud"],
    columns=["Predicted Legitimate", "Predicted Fraud"]
)

cm_rf_df

,Predicted Legitimate,Predicted Fraud
Actual Legitimate,119168,677
Actual Fraud,26,129


# **PHASE 9 - Model comparison table**

In [207]:
from sklearn.metrics import precision_score, recall_score, f1_score

results = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Decision Tree",
        "Random Forest"
    ],

    "Precision": [
        precision_score(y_val, y_val_pred_lr),
        precision_score(y_val, y_val_pred_dt),
        precision_score(y_val, y_val_pred_rf)
    ],

    "Recall": [
        recall_score(y_val, y_val_pred_lr),
        recall_score(y_val, y_val_pred_dt),
        recall_score(y_val, y_val_pred_rf)
    ],

    "F1": [
        f1_score(y_val, y_val_pred_lr),
        f1_score(y_val, y_val_pred_dt),
        f1_score(y_val, y_val_pred_rf)
    ],

    "ROC_AUC": [
        roc_auc_score(y_val, y_val_prob_lr),
        roc_auc_score(y_val, y_val_prob_dt),
        roc_auc_score(y_val, y_val_prob_rf)
    ],

    "PR_AUC": [
        average_precision_score(y_val, y_val_prob_lr),
        average_precision_score(y_val, y_val_prob_dt),
        average_precision_score(y_val, y_val_prob_rf)
    ]
})

results.sort_values("PR_AUC", ascending=False)

,Model,Precision,Recall,F1,ROC_AUC,PR_AUC
2,Random Forest,0.160050,0.832258,0.268470,0.992122,0.748048
1,Decision Tree,0.034313,0.941935,0.066213,0.967899,0.681471
0,Logistic Regression,0.015390,0.929032,0.030278,0.978335,0.247458


# **PHASE 10 - Imbalance Experiment**

Fraud is extremely rare in the dataset, so the model is evaluated under different imbalance-handling strategies.

The current models use class weighting. This experiment compares class weighting against random undersampling of the majority class.

In [208]:
from imblearn.under_sampling import RandomUnderSampler

rus = RandomUnderSampler(
    sampling_strategy=0.1,
    random_state=42
)

X_train_under, y_train_under = rus.fit_resample(
    X_train_processed,
    y_train
)

print("Original training distribution:")
print(y_train.value_counts())

print("\nAfter undersampling:")
print(pd.Series(y_train_under).value_counts())

Original training distribution:
isFraud
0    559277
1       723
Name: count, dtype: int64

After undersampling:
isFraud
0    7230
1     723
Name: count, dtype: int64


In [209]:
rf_undersampled = RandomForestClassifier(
    n_estimators=100,
    max_depth=15,
    min_samples_leaf=5,
    n_jobs=-1,
    random_state=42
)

rf_undersampled.fit(X_train_under, y_train_under)

RandomForestClassifier(max_depth=15, min_samples_leaf=5, n_jobs=-1,
                       random_state=42)

In [210]:
y_val_pred_rf_under = rf_undersampled.predict(X_val_processed)
y_val_prob_rf_under = rf_undersampled.predict_proba(X_val_processed)[:, 1]

print("RANDOM FOREST - UNDERSAMPLED")
print(classification_report(y_val, y_val_pred_rf_under))

print("Confusion Matrix:")
print(confusion_matrix(y_val, y_val_pred_rf_under))

print("\nROC-AUC:",
      roc_auc_score(y_val, y_val_prob_rf_under))

print("PR-AUC:",
      average_precision_score(y_val, y_val_prob_rf_under))

RANDOM FOREST - UNDERSAMPLED
              precision    recall  f1-score   support

           0       1.00      0.99      1.00    119845
           1       0.17      0.87      0.29       155

    accuracy                           0.99    120000
   macro avg       0.59      0.93      0.64    120000
weighted avg       1.00      0.99      1.00    120000

Confusion Matrix:
[[119193    652]
 [    20    135]]

ROC-AUC: 0.9924201017712394
PR-AUC: 0.705096722498554


In [211]:
imbalance_results = pd.DataFrame({
    "Strategy": [
        "Class Weighting",
        "Random Undersampling"
    ],
    "Precision": [
        precision_score(y_val, y_val_pred_rf),
        precision_score(y_val, y_val_pred_rf_under)
    ],
    "Recall": [
        recall_score(y_val, y_val_pred_rf),
        recall_score(y_val, y_val_pred_rf_under)
    ],
    "F1": [
        f1_score(y_val, y_val_pred_rf),
        f1_score(y_val, y_val_pred_rf_under)
    ],
    "PR_AUC": [
        average_precision_score(y_val, y_val_prob_rf),
        average_precision_score(y_val, y_val_prob_rf_under)
    ]
})

imbalance_results

,Strategy,Precision,Recall,F1,PR_AUC
0,Class Weighting,0.160050,0.832258,0.268470,0.748048
1,Random Undersampling,0.171537,0.870968,0.286624,0.705097


In [212]:
experiment_summary = pd.DataFrame({
    "Experiment": [
        "Logistic Regression",
        "Decision Tree",
        "Random Forest - Class Weighting",
        "Random Forest - Undersampling"
    ],
    "Purpose": [
        "Baseline",
        "Tree comparison",
        "Ensemble comparison",
        "Imbalance experiment"
    ],
    "Precision": [
        precision_score(y_val, y_val_pred_lr),
        precision_score(y_val, y_val_pred_dt),
        precision_score(y_val, y_val_pred_rf),
        precision_score(y_val, y_val_pred_rf_under)
    ],
    "Recall": [
        recall_score(y_val, y_val_pred_lr),
        recall_score(y_val, y_val_pred_dt),
        recall_score(y_val, y_val_pred_rf),
        recall_score(y_val, y_val_pred_rf_under)
    ],
    "F1": [
        f1_score(y_val, y_val_pred_lr),
        f1_score(y_val, y_val_pred_dt),
        f1_score(y_val, y_val_pred_rf),
        f1_score(y_val, y_val_pred_rf_under)
    ],
    "PR-AUC": [
        average_precision_score(y_val, y_val_prob_lr),
        average_precision_score(y_val, y_val_prob_dt),
        average_precision_score(y_val, y_val_prob_rf),
        average_precision_score(y_val, y_val_prob_rf_under)
    ]
})

experiment_summary

,Experiment,Purpose,Precision,Recall,F1,PR-AUC
0,Logistic Regression,Baseline,0.015390,0.929032,0.030278,0.247458
1,Decision Tree,Tree comparison,0.034313,0.941935,0.066213,0.681471
2,Random Forest - Class Weighting,Ensemble comparison,0.160050,0.832258,0.268470,0.748048
3,Random Forest - Undersampling,Imbalance experiment,0.171537,0.870968,0.286624,0.705097


### Final Model Selection

Random undersampling achieved slightly higher recall and F1-score at the default 0.50 threshold.

However, the class-weighted Random Forest achieved the stronger PR-AUC, indicating better overall precision-recall ranking performance across thresholds.

Because the project also includes threshold tuning, the class-weighted Random Forest was selected as the final model and its operating threshold was optimized separately on the validation set.

# **PHASE 11 - Threshold Tuning**

The default classification threshold of 0.50 may not be optimal for fraud detection.

Different thresholds are evaluated to study the trade-off between fraud recall and false-positive rate.

In [213]:
thresholds = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7]

threshold_results = []

for threshold in thresholds:
    preds = (y_val_prob_rf >= threshold).astype(int)

    threshold_results.append({
        "Threshold": threshold,
        "Precision": precision_score(y_val, preds),
        "Recall": recall_score(y_val, preds),
        "F1": f1_score(y_val, preds)
    })

threshold_df = pd.DataFrame(threshold_results)

threshold_df

,Threshold,Precision,Recall,F1
0,0.1,0.027840,0.961290,0.054113
1,0.2,0.046741,0.948387,0.089091
2,0.3,0.069258,0.896774,0.128585
3,0.4,0.105223,0.883871,0.188058
4,0.5,0.160050,0.832258,0.268470
5,0.6,0.289786,0.787097,0.423611
6,0.7,0.678571,0.735484,0.705882


**Then choose the threshold with highest F1**

In [214]:
best_row = threshold_df.loc[threshold_df["F1"].idxmax()]

best_threshold = best_row["Threshold"]

print("Best threshold:", best_threshold)
print(best_row)

Best threshold: 0.7
Threshold    0.700000
Precision    0.678571
Recall       0.735484
F1           0.705882
Name: 6, dtype: float64


### Threshold Selection Objective

Fraud detection involves a trade-off between false negatives and false positives.

- False negatives represent fraudulent transactions that remain undetected and may cause financial loss.
- False positives represent legitimate transactions unnecessarily sent for investigation, increasing operational workload.

The operating threshold was therefore selected using validation data only, rather than automatically using 0.50.

Among the tested thresholds, 0.70 achieved the strongest F1-score, providing a more balanced operating point between fraud recall and false-positive control.

The final test set remained untouched until the threshold had been selected.

# **PHASE 12 - Final Test Evaluation**

The final selected model is evaluated once on the untouched test set using the threshold selected from the validation set.

In [215]:
y_test_prob = random_forest_model.predict_proba(X_test_processed)[:, 1]

y_test_pred = (y_test_prob >= best_threshold).astype(int)

In [216]:
print("FINAL TEST RESULTS")
print()

print(classification_report(y_test, y_test_pred))

print("\nROC-AUC:",
      roc_auc_score(y_test, y_test_prob))

print("PR-AUC:",
      average_precision_score(y_test, y_test_prob))

FINAL TEST RESULTS

              precision    recall  f1-score   support

           0       1.00      1.00      1.00    119845
           1       0.60      0.68      0.64       155

    accuracy                           1.00    120000
   macro avg       0.80      0.84      0.82    120000
weighted avg       1.00      1.00      1.00    120000


ROC-AUC: 0.9936926594700951
PR-AUC: 0.7174217145459886


In [217]:
cm = confusion_matrix(y_test, y_test_pred)

cm_df = pd.DataFrame(
    cm,
    index=["Actual Legitimate", "Actual Fraud"],
    columns=["Predicted Legitimate", "Predicted Fraud"]
)

cm_df

,Predicted Legitimate,Predicted Fraud
Actual Legitimate,119774,71
Actual Fraud,49,106


# **PHASE 13 - Count frauds missed**

In [218]:
cm = confusion_matrix(y_test, y_test_pred)

tn, fp, fn, tp = cm.ravel()

final_cm_df = pd.DataFrame({
    "Metric": [
        "True Negatives",
        "False Positives",
        "False Negatives (Frauds Missed)",
        "True Positives (Frauds Detected)"
    ],
    "Count": [tn, fp, fn, tp]
})

final_cm_df

,Metric,Count
0,True Negatives,119774
1,False Positives,71
2,False Negatives (Frauds Missed),49
3,True Positives (Frauds Detected),106


# **PHASE 14 - Error Analysis**

The final model is analyzed by inspecting false positives and false negatives.

- False positives are legitimate transactions incorrectly flagged as fraud.
- False negatives are fraudulent transactions missed by the model.

For fraud detection, false negatives are particularly important because they represent fraudulent transactions that would pass through the system undetected.

In [219]:
test_results = X_test.copy()

test_results["Actual"] = y_test.values
test_results["Predicted"] = y_test_pred
test_results["Fraud_Probability"] = y_test_prob

test_results.head()

,step,type,amount,oldbalanceOrg,oldbalanceDest,Actual,Predicted,Fraud_Probability
4528551,326,CASH_IN,183936.29,18889151.04,1064576.46,0,0,0.0
23628,8,CASH_OUT,63216.18,0.00,290379.23,0,0,0.0
349437,17,CASH_OUT,153547.07,0.00,233361.29,0,0,0.0
4406678,322,CASH_IN,278986.94,4807028.17,4724795.18,0,0,0.0
6184749,567,CASH_OUT,227185.83,0.00,279511.50,0,0,0.0


In [220]:
false_positives = test_results[
    (test_results["Actual"] == 0) &
    (test_results["Predicted"] == 1)
]

false_negatives = test_results[
    (test_results["Actual"] == 1) &
    (test_results["Predicted"] == 0)
]

print("False Positives:", len(false_positives))
print("False Negatives:", len(false_negatives))

False Positives: 71
False Negatives: 49


In [221]:
false_positives.head(10)

,step,type,amount,oldbalanceOrg,oldbalanceDest,Actual,Predicted,Fraud_Probability
4975249,352,TRANSFER,1377272.64,981749.00,0.00,0,1,0.828945
1986747,179,CASH_OUT,558842.50,600808.00,0.00,0,1,0.731450
4071154,300,CASH_OUT,243360.59,237856.00,246877.72,0,1,0.739927
5353859,375,TRANSFER,404662.27,406586.00,0.00,0,1,0.775690
4297507,307,TRANSFER,192526.46,193194.00,0.00,0,1,0.924761
2713197,211,CASH_OUT,226563.08,233837.00,396461.54,0,1,0.728793
4407326,322,CASH_OUT,199654.60,187187.58,684376.38,0,1,0.718611
3282366,251,TRANSFER,156409.82,217452.00,0.00,0,1,0.715470
4209746,305,CASH_OUT,276463.77,282530.00,3131425.74,0,1,0.724321
4403407,321,CASH_OUT,262219.79,237267.00,624517.75,0,1,0.760212


In [222]:
false_negatives.head(10)

,step,type,amount,oldbalanceOrg,oldbalanceDest,Actual,Predicted,Fraud_Probability
5149974,357,CASH_OUT,74921.74,74921.74,568763.30,1,0,0.234059
1325512,137,CASH_OUT,417217.89,417217.89,2468054.62,1,0,0.582741
1932232,176,CASH_OUT,102229.91,102229.91,0.00,1,0,0.342706
58307,9,TRANSFER,21922.00,21922.00,0.00,1,0,0.386582
1030398,61,CASH_OUT,55062.46,55062.46,0.00,1,0,0.370609
1932238,176,CASH_OUT,665489.33,665489.33,1184319.45,1,0,0.689770
6259933,604,CASH_OUT,714218.48,714218.48,106940.83,1,0,0.697728
5782686,400,CASH_OUT,460689.45,460689.45,0.00,1,0,0.567619
5991575,419,CASH_OUT,149535.00,149535.00,711124.24,1,0,0.571830
653948,35,TRANSFER,136983.26,136983.26,0.00,1,0,0.506594


### Error Analysis Findings

The final model produced 71 false positives and 49 false negatives.

Both error types were concentrated in the `CASH_OUT` and `TRANSFER` transaction classes, which are also the only transaction types containing fraud cases in this dataset.

False positives had a median predicted fraud probability of approximately 0.73, meaning many of them were close to the selected 0.70 decision threshold.

False negatives had a lower median fraud probability of approximately 0.51, suggesting that some fraudulent transactions receive only moderate model confidence and are therefore missed at the selected operating threshold.

This highlights the trade-off between fraud recall and false-positive review workload. The model should therefore be used as a decision-support tool rather than an automatic transaction-blocking system.

In [223]:
error_pattern_summary = pd.DataFrame({
    "Group": [
        "False Positives",
        "False Negatives"
    ],
    "Count": [
        len(false_positives),
        len(false_negatives)
    ],
    "Median Amount": [
        false_positives["amount"].median(),
        false_negatives["amount"].median()
    ],
    "Median Fraud Probability": [
        false_positives["Fraud_Probability"].median(),
        false_negatives["Fraud_Probability"].median()
    ],
    "Most Common Type": [
        false_positives["type"].mode().iloc[0],
        false_negatives["type"].mode().iloc[0]
    ]
})

error_pattern_summary

,Group,Count,Median Amount,Median Fraud Probability,Most Common Type
0,False Positives,71,226563.08,0.731450,CASH_OUT
1,False Negatives,49,108755.28,0.506594,CASH_OUT


In [224]:
false_positives["type"].value_counts()

type
CASH_OUT    52
TRANSFER    19
Name: count, dtype: int64

In [225]:
false_negatives["type"].value_counts()

type
CASH_OUT    39
TRANSFER    10
Name: count, dtype: int64

# **PHASE 15 - Save Final Model Artifacts**

## **Final Reusable Inference Pipeline**

The fitted preprocessing transformer and selected Random Forest classifier are combined into a single reusable inference pipeline.

The preprocessing transformer was fitted only on the training data before being combined with the classifier, preserving the leakage-safe workflow.

In [226]:
from sklearn.pipeline import Pipeline

final_inference_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", random_forest_model)
    ]
)

final_inference_pipeline

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('num', StandardScaler(),
                                                  ['step', 'amount',
                                                   'oldbalanceOrg',
                                                   'oldbalanceDest']),
                                                 ('cat',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['type'])])),
                ('classifier',
                 RandomForestClassifier(class_weight='balanced', max_depth=15,
                                        min_samples_leaf=5, n_jobs=-1,
                                        random_state=42))])

In [227]:
pipeline_test_prob = final_inference_pipeline.predict_proba(
    X_test.head(5)
)[:, 1]

pipeline_test_prob

array([0., 0., 0., 0., 0.])

In [228]:
joblib.dump(
    final_inference_pipeline,
    "/kaggle/working/fraud_pipeline.joblib"
)

print("Final reusable pipeline saved successfully.")

Final reusable pipeline saved successfully.


In [229]:
import joblib

joblib.dump(
    preprocessor,
    "/kaggle/working/preprocessor.joblib"
)

joblib.dump(
    random_forest_model,
    "/kaggle/working/fraud_model.joblib"
)

joblib.dump(
    best_threshold,
    "/kaggle/working/threshold.joblib"
)

print("Model artifacts saved successfully.")

Model artifacts saved successfully.


# **PHASE 16 - Inference Function**

The inference function receives one transaction, applies the saved preprocessing workflow, generates a fraud probability, and converts it into a prediction and risk level.

In [230]:
def predict_fraud(transaction):
    
    transaction_df = pd.DataFrame([transaction])

    transaction_processed = preprocessor.transform(transaction_df)

    fraud_probability = random_forest_model.predict_proba(
        transaction_processed
    )[:, 1][0]

    prediction = int(
        fraud_probability >= best_threshold
    )

    if fraud_probability < 0.30:
        risk_level = "Low"
    elif fraud_probability < best_threshold:
        risk_level = "Medium"
    else:
        risk_level = "High"

    return {
        "fraud_probability": round(float(fraud_probability), 4),
        "prediction": prediction,
        "risk_level": risk_level,
        "threshold": float(best_threshold)
    }

In [231]:
sample_transaction = {
    "step": 1,
    "type": "TRANSFER",
    "amount": 181.0,
    "oldbalanceOrg": 181.0,
    "oldbalanceDest": 0.0
}

predict_fraud(sample_transaction)

{'fraud_probability': 0.4375,
 'prediction': 0,
 'risk_level': 'Medium',
 'threshold': 0.7}

## Saved Artifact Loading Test

The saved preprocessing transformer, final Random Forest model, and selected threshold are reloaded from disk to verify that the deployment workflow uses the same artifacts produced during training.

In [232]:
loaded_preprocessor = joblib.load("/kaggle/working/preprocessor.joblib")
loaded_model = joblib.load("/kaggle/working/fraud_model.joblib")
loaded_threshold = joblib.load("/kaggle/working/threshold.joblib")

In [233]:
def predict_fraud_saved(transaction):

    transaction_df = pd.DataFrame([transaction])

    transaction_processed = loaded_preprocessor.transform(transaction_df)

    fraud_probability = loaded_model.predict_proba(
        transaction_processed
    )[:, 1][0]

    prediction = int(
        fraud_probability >= loaded_threshold
    )

    if fraud_probability < 0.30:
        risk_level = "Low"
    elif fraud_probability < loaded_threshold:
        risk_level = "Medium"
    else:
        risk_level = "High"

    return {
        "fraud_probability": round(float(fraud_probability), 4),
        "prediction": prediction,
        "risk_level": risk_level,
        "threshold": float(loaded_threshold)
    }

In [234]:
predict_fraud_saved(sample_transaction)

{'fraud_probability': 0.4375,
 'prediction': 0,
 'risk_level': 'Medium',
 'threshold': 0.7}